# Quickstart

Profile an orders table, retrieve a finding, inspect its evidence, and check
what work supports it. This fixture contains a primary key, a region
lookup, a nearly unique external reference with two repeated values, and a
derived quantity. The known structure makes the result easy to check.

The basic workflow is `report = onceover.profile(df)`, followed by
`report.select(...)` for complete findings and `report.coverage` for the
scope of the search. The [capability guide](capabilities.ipynb) introduces
the other questions onceover can answer.


In [1]:
import json
import random

import polars as pl

import onceover

rng = random.Random(7)
n = 2000

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
order_id = list(range(n))
external_ref = [f"R{i:05}" for i in range(n)]
external_ref[-1] = external_ref[0]   # two duplicated values
external_ref[-2] = external_ref[1]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]

df = pl.DataFrame({
    "order_id": order_id,
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
})
from great_tables import GT, loc, style
(GT(df.head(4))
 .tab_header(title="Four rows from the orders fixture")
 .tab_options(table_width="100%", table_layout="fixed")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere;"), loc.body()))

GT(_tbl_data=shape: (4, 6)
┌──────────┬──────────────┬───────────┬─────────────┬──────────┬─────────────────┐
│ order_id ┆ external_ref ┆ region_id ┆ region_name ┆ quantity ┆ double_quantity │
│ ---      ┆ ---          ┆ ---       ┆ ---         ┆ ---      ┆ ---             │
│ i64      ┆ str          ┆ i64       ┆ str         ┆ i64      ┆ i64             │
╞══════════╪══════════════╪═══════════╪═════════════╪══════════╪═════════════════╡
│ 0        ┆ R00000       ┆ 1         ┆ amer        ┆ 1        ┆ 2               │
│ 1        ┆ R00001       ┆ 0         ┆ emea        ┆ 7        ┆ 14              │
│ 2        ┆ R00002       ┆ 1         ┆ amer        ┆ 7        ┆ 14              │
│ 3        ┆ R00003       ┆ 2         ┆ apac        ┆ 4        ┆ 8               │
└──────────┴──────────────┴───────────┴─────────────┴──────────┴─────────────────┘, _body=<great_tables._gt_data.Body object at 0x71f5c1350470>, _boxhead=Boxhead([ColInfo(var='order_id', type=<ColInfoTypeEnum.default: 1>, column_label='order_id', column_align='right', column_width=None, stub_level=None), ColInfo(var='external_ref', type=<ColInfoTypeEnum.default: 1>, column_label='external_ref', column_align='left', column_width=None, stub_level=None), ColInfo(var='region_id', type=<ColInfoTypeEnum.default: 1>, column_label='region_id', column_align='right', column_width=None, stub_level=None), ColInfo(var='region_name', type=<ColInfoTypeEnum.default: 1>, column_label='region_name', column_align='left', column_width=None, stub_level=None), ColInfo(var='quantity', type=<ColInfoTypeEnum.default: 1>, column_label='quantity', column_align='right', column_width=None, stub_level=None), ColInfo(var='double_quantity', type=<ColInfoTypeEnum.default: 1>, column_label='double_quantity', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x71f5a9cbafc0>, _spanners=Spanners([]), _heading=Heading(title='Four rows from the orders fixture', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x71f5a833b5f0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x71f5c12e0230>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='order_id', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='order_id', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='order_id', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='order_id', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='external_ref', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='external_ref', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='external_ref', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='external_ref', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='region_id', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap

## Profile it

One call runs the detectors enabled by the default budget.
[](large_tables.ipynb) describes runtime and coverage on larger tables.

In [2]:
report = onceover.profile(df)
print(report.overview["headline"])

2000 rows × 6 columns (standard budget); key candidate: order_id; 0 functional dependencies, 4 predictive dependencies, 16 conditional rules


The headline reports a key candidate, functional dependencies,
predictable columns, and conditional rules for this fixture.

## Inspect findings

`report.findings` is sorted by its priority score, `interestingness`:


In [3]:
finding_index = report.findings_frame().select("category", "type", "claim").head(8)
(GT(finding_index)
 .tab_header(title="Highest-ranked findings", subtitle="Complete claims; retrieve a finding for all its evidence")
 .tab_options(table_width="100%", table_layout="fixed", table_font_size="14px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))

GT(_tbl_data=shape: (8, 3)
┌───────────────────┬────────────┬─────────────────────────────────┐
│ category          ┆ type       ┆ claim                           │
│ ---               ┆ ---        ┆ ---                             │
│ str               ┆ str        ┆ str                             │
╞═══════════════════╪════════════╪═════════════════════════════════╡
│ uniqueness        ┆ uniqueness ┆ order_id uniquely identifies r… │
│ uniqueness        ┆ uniqueness ┆ column external_ref is nearly … │
│ dependencies      ┆ affine     ┆ double_quantity = 2.000000·qua… │
│ dependencies      ┆ predictive ┆ region_id is predictable from … │
│ conditional_rules ┆ if_then    ┆ When double_quantity <= 2, qua… │
│ conditional_rules ┆ if_then    ┆ When quantity <= 1, double_qua… │
│ conditional_rules ┆ if_then    ┆ When double_quantity > 10 and … │
│ conditional_rules ┆ if_then    ┆ When double_quantity > 18 and … │
└───────────────────┴────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x71f5a9d280b0>, _boxhead=Boxhead([ColInfo(var='category', type=<ColInfoTypeEnum.default: 1>, column_label='category', column_align='left', column_width=None, stub_level=None), ColInfo(var='type', type=<ColInfoTypeEnum.default: 1>, column_label='type', column_align='left', column_width=None, stub_level=None), ColInfo(var='claim', type=<ColInfoTypeEnum.default: 1>, column_label='claim', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x71f5c12e1820>, _spanners=Spanners([]), _heading=Heading(title='Highest-ranked findings', subtitle='Complete claims; retrieve a finding for all its evidence', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x71f59b73de50>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x71f59b6b25d0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='category', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='category', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='category', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='category', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='category', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='category', rownum=5, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='category', rownum=6, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='category', rownum=7, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='type', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='type', rownum=1, colnum=None, styles=[CellStyleCss

The report verifies mappings between `quantity` and `double_quantity`,
checks their affine formula, and finds the region lookup. It also reports
that `order_id` is unique while `external_ref` has two repeated values.

The following count shows the categories present in this report:


In [4]:
from collections import Counter

print(Counter((f["category"], f.get("type")) for f in report.findings))

Counter({('conditional_rules', 'if_then'): 16, ('associations', 'level_contrast'): 12, ('dependencies', 'predictive'): 4, ('column_roles', 'categorical_role'): 3, ('uniqueness', 'uniqueness'): 2, ('column_roles', 'duplicate_encoding'): 2, ('associations', 'statistical'): 2, ('concentration', 'hotspot'): 2, ('column_roles', 'identifier_role'): 2, ('dependencies', 'affine'): 1, ('dependencies', 'order_constraint'): 1, ('ordering', 'column_ordering'): 1})


To inspect one column, use `report.select(column=...)`. The returned
entries are complete findings, in report order:

In [5]:
for finding in report.select(column="external_ref"):
    print(f"[{finding['category']}] {finding['claim']}")

[uniqueness] column external_ref is nearly unique with 2 repeated non-null values
[column_roles] column external_ref looks like a unique identifier (1998 distinct of 2000 values)


`external_ref` looks like an identifier, but it contains two repeated
values. Joining on it as though it were unique can produce extra rows.
Inspect the uniqueness finding before choosing it as a key.

## Inspect a complete finding


In [6]:
finding = report.select(category="uniqueness", column="external_ref")[0]
print(json.dumps(finding, indent=2))

{
  "id": "uniqueness-c4f85703f4a943ff",
  "category": "uniqueness",
  "facets": {
    "scope": "column",
    "object": "relation",
    "support": "exact",
    "conditionality": "unconditional",
    "temporality": "atemporal",
    "quality_dimension": "uniqueness",
    "status": "regularity"
  },
  "type": "uniqueness",
  "subject": {
    "columns": [
      "external_ref"
    ],
    "unit": "column_values"
  },
  "specification": {
    "definition": {
      "comparison": "canonical_equality"
    }
  },
  "claim": "column external_ref is nearly unique with 2 repeated non-null values",
  "evidence": [
    {
      "method": {
        "detector": "keys",
        "algorithm": "near_uniqueness",
        "provenance": "exact distinct-count analysis (onceover v0.1)"
      },
      "basis": "measured",
      "scope": {
        "sampling": "full",
        "population_rows": 2000,
        "rows_evaluated": 2000,
        "missing_values": "non-missing values counted; missingness reported separatel

The result names its columns in `subject`. `specification` describes the
comparison, mapping, or formula, and `evidence` contains the evaluated row
counts and metrics. The [findings chapter](findings.ipynb) explains these
fields and shows how to filter on them.


In [7]:
identity = next(f for f in report.select(category="dependencies", type="affine")
                if f["specification"].get("form") == "affine")
print(identity["claim"])
print("category:", identity["category"], "| type:", identity.get("type"))
print("form:", identity["specification"].get("form"))
ev = identity["evidence"][0]
print("basis:", ev["basis"], "| sampling:", ev["scope"]["sampling"])
print("validation:", ev["validation"]["method"])
print("metrics:", ev["metrics"])

double_quantity = 2.000000·quantity within tolerance on all 2000 non-missing row pairs
category: dependencies | type: affine
form: affine
basis: measured | sampling: full
validation: direct_comparison
metrics: {'a': 2.0, 'absolute_tolerance': 1e-09, 'b': 0.0, 'exact_fraction': 1.0, 'matching_rows': 2000, 'n': 2000, 'relative_tolerance': 1e-09, 'tolerance': '1e-9 absolute + 1e-9 relative', 'violations': 0}


The identity check fitted `y = a*x + b` on a sample, then checked the
residuals on all 2,000 rows. Every row matched within the stated tolerance.

## Inspect column summaries

`report.columns` contains counts, statistics, and suggested roles, keyed
by column name. For example:


In [8]:
print(json.dumps(report.columns["region_id"], indent=2))
print(json.dumps(report.columns["double_quantity"]["numeric_stats"], indent=2))

{
  "name": "region_id",
  "dtype": "int64",
  "card_role": "categorical",
  "semantic_role": "dimension",
  "semantic_role_evidence": "3 distinct values",
  "n": 2000,
  "null_count": 0,
  "null_fraction": 0.0,
  "distinct": 3,
  "distinct_ratio": 0.0015,
  "numeric_stats": {
    "min": 0.0,
    "max": 2.0,
    "mean": 0.952,
    "std": 0.816514543654919,
    "q25": 0.0,
    "q50": 1.0,
    "q75": 2.0,
    "zeros": 717,
    "negatives": 0,
    "integral": true
  },
  "concentration": {
    "hhi": 0.3344935,
    "entropy_nats": 1.0968765707316228,
    "effective_cardinality": 2.994797362652193,
    "top1_share": 0.3585,
    "top3_share": 1.0
  },
  "top_values": [
    {
      "value": "0",
      "count": 717,
      "share": 0.3585
    },
    {
      "value": "1",
      "count": 662,
      "share": 0.331
    },
    {
      "value": "2",
      "count": 621,
      "share": 0.3105
    }
  ],
  "action_hints": [
    "treat_as_categorical"
  ]
}
{
  "min": 2.0,
  "max": 20.0,
  "mean": 10.92

`region_id` has three distinct numeric values and receives a categorical
role with a `treat_as_categorical` suggestion. `double_quantity` includes
exact distribution summaries. These counts use every input row; see
[Column summaries](columns.ipynb) for the available fields.

## Check which passes ran

`report.coverage` lists completed and skipped passes, along with their
sample sizes and limits:


In [9]:
coverage_rows = [{"Pass": name, "Status": info["status"], "Detail": info["detail"]}
                 for name, info in report.coverage.items()]
(GT(pl.DataFrame(coverage_rows))
 .tab_header(title="What the profile examined")
 .tab_options(table_width="100%", table_layout="fixed", table_font_size="14px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))

GT(_tbl_data=shape: (54, 3)
┌───────────────────────┬─────────────┬─────────────────────────────────┐
│ Pass                  ┆ Status      ┆ Detail                          │
│ ---                   ┆ ---         ┆ ---                             │
│ str                   ┆ str         ┆ str                             │
╞═══════════════════════╪═════════════╪═════════════════════════════════╡
│ arithmetic_identities ┆ ran_exact   ┆ 1 candidate pairs checked, 1 i… │
│ association_pairs     ┆ ran_sampled ┆ 0 nominal pairs, 4 mixed pairs… │
│ association_rules     ┆ ran_sampled ┆ 1 of 1 nominal features (all e… │
│ boundary_mass         ┆ ran_exact   ┆ 4 numeric columns scanned for … │
│ breaks                ┆ ran_sampled ┆ 3 of 3 numeric series scanned … │
│ …                     ┆ …           ┆ …                               │
│ sentinels             ┆ ran_exact   ┆ 4 numeric columns scanned for … │
│ series                ┆ skipped     ┆ no date or datetime axis to de… │
│ slice_shifts          ┆ skipped     ┆ on-request detector; not in th… │
│ sortedness            ┆ ran_exact   ┆ 1 columns sorted by row order   │
│ target_shapes         ┆ skipped     ┆ runs when a target is designat… │
└───────────────────────┴─────────────┴─────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x71f5a9eb5a60>, _boxhead=Boxhead([ColInfo(var='Pass', type=<ColInfoTypeEnum.default: 1>, column_label='Pass', column_align='left', column_width=None, stub_level=None), ColInfo(var='Status', type=<ColInfoTypeEnum.default: 1>, column_label='Status', column_align='left', column_width=None, stub_level=None), ColInfo(var='Detail', type=<ColInfoTypeEnum.default: 1>, column_label='Detail', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x71f59b6b31a0>, _spanners=Spanners([]), _heading=Heading(title='What the profile examined', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x71f59b6b2630>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x71f59b6b3230>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=5, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=6, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=7, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Pass', rownum=8, colnum=None, styles=[CellStyleCss(rule

`row_duplicates` can rule out duplicate rows using the verified unique,
null-free `order_id`. `functional_dependencies` reports its verification
sample. This fixture fits within that cap; larger inputs may be sampled.

## Declare what you know

A supplied calendar makes a gap check explicit. Here March is absent from
a monthly series that is too short for the inferred date detector. Hints
provide the expected cadence; the `expectations` evidence records
`reference_source="declared"`.


In [10]:
import datetime

monthly = pl.DataFrame({
    "month": [datetime.date(2024, m, 1) for m in (1, 2, 4, 5, 6)],
    "revenue": [10.0, 11.0, 13.0, 12.0, 14.0],
})
hinted = onceover.profile(monthly, hints={
    "time": "month",
    "no_gaps": ["month"],
    "cadence": {"month": "monthly"},
})
for f in hinted.findings:
    if f["category"] == "gaps":
        print(f["claim"])
        print(f["evidence"][0]["metrics"])

column month is missing 1 month at 2024-02-01..2024-04-01 (60 days elapsed)
{'approximate': False, 'elapsed_days': 60, 'from': '2024-02-01', 'missing': 1, 'n': 5, 'reference': 'months', 'reference_source': 'declared', 'to': '2024-04-01', 'triviality': 'scaffolding_subject', 'triviality_factor': 0.6}


## Save the report

`save()` writes the complete JSON report by default. For repeatable JSON,
use the same package version, data, and settings with no deadline and with
`timing=False`. A short digest records its omissions and links back to the
full report through finding IDs.


In [11]:
import os
import tempfile

path = report.save(tempfile.mkdtemp() + "/profile.json")
print("saved", os.path.getsize(path), "bytes to", path)
print(report.to_digest(800))

saved 130532 bytes to /tmp/tmpuis9ww76/profile.json
profile digest: 2000 rows x 6 columns, 48 findings, schema 0.8
account: order_id uniquely identifies rows
headline: 2000 rows × 6 columns (standard budget); key candidate: order_id; 0 functional dependencies, 4 predictive dependencies, 16 conditional rules
... [19 rolled-up finding groups omitted to fit 800 chars; 48 findings total: conditional_rules 16, associations 14, column_roles 7, dependencies 6; report.select(category="conditional_rules") retrieves them]



`save()` writes JSON by default; `report.data` returns the parsed report.
`to_markdown()` produces a text summary suitable for sharing in an issue
or pull request.

## Further examples

- [Reading findings](findings.ipynb): fields, scores, and filtering.
- [Column summaries](columns.ipynb): roles and distribution statistics.
- [Coverage](coverage.ipynb): budgets, sampling, and detector selection.
- [Analysis notebook](https://github.com/fcela/onceover/blob/main/notebooks/onceover_analysis.ipynb):
  a larger fixture with charts and checks against planted relationships.


For the full method map, continue with [capabilities](capabilities.ipynb).
The [API reference](api.ipynb) covers targeting, hints, report comparison,
size-budgeted JSON, and per-pass timing.
